In [ ]:
# Optional manual presets
# Leave these as None to use the interactive folder and file pickers.
PRESET_RAW_DIR = None        # Raw NDA folder
PRESET_META_XLSX = None      # Metadata workbook
PRESET_CYCLE_ROUTINE = None  # Cycle map workbook
PRESET_OUTPUT_DIR = None     # Output folder
PRESET_OUTPUT_NAME = None    # Output filename without extension
PRESET_CUSTOM_ACR = None     # Optional ACR value


# Cycling Processor

- Author: Minsung Baek
- Created: 2022-08-03
- Updated: 2025-07-07

Overview: This notebook manually processes cycling NDA files into an Excel workbook with charts and a .stage1ready.pkl file for downstream Stage 1 extraction. Select the raw NDA folder, metadata workbook, cycle map workbook, and output folder when prompted. NDA filenames and metadata barcodes should match so each file can be mapped to the correct cell and group.


# Select Directory where files are located

In [ ]:
import traitlets
from ipywidgets import widgets
from IPython.display import display
from tkinter import Tk, filedialog
import pandas as pd
import numpy as np
import os
from scipy.stats import linregress 
from scipy.interpolate import interp1d
import math
import xlsxwriter
import matplotlib
import regex as re
import NewareNDA
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

def get_statis(df_detail):
    """
    A function to extract the statis sheet from the detail sheet
    df_statis is a higher-level dataframe.It summarizes the data of each step in df_detail 
    """
    df_detail_group = df_detail.groupby('Step')
    df_statis = pd.concat([df_detail_group['Cycle'].last().to_frame(), # cycle number from Neware
                           df_detail_group['Current(mA)'].first().to_frame(), # start current
                           df_detail_group['Current(mA)'].last().to_frame(),# end current
                           df_detail_group['Voltage'].first().to_frame(),# start voltage
                           df_detail_group['Voltage'].last().to_frame(), # end voltage             
                           df_detail_group['Status'].first().to_frame(), # step condition  
                           df_detail_group['Time'].last().to_frame(), # step time                        
                           df_detail_group['Charge_Capacity(mAh)'].last().to_frame(),                           
                           df_detail_group['Discharge_Capacity(mAh)'].last().to_frame(),                           
                           df_detail_group['Charge_Energy(mWh)'].last().to_frame(),                           
                           df_detail_group['Discharge_Energy(mWh)'].last().to_frame(),  
                           df_detail_group['Timestamp'].first().to_frame(),# start time
                           df_detail_group['Timestamp'].last().to_frame(), # end time
                           #df_detail_group['dV/dt_MA'].last().to_frame(),   # moving average of dV/dt at the end of step                              
                           ],
                          axis=1)    
    # update more column names    
    df_statis.columns = ['Cycle',
                        'Start Current(mA)', 'End Current(mA)',
                        'Start Voltage(V)', 'End Voltage(V)',
                        'Status', 'Step time',
                        'Charge Capacity(mAh)', 'Discharge Capacity(mAh)',
                        'Charge Energy(mWh)', 'Discharge Energy(mWh)',
                        'Start time', 'End time'
                        ]
    df_statis.reset_index(inplace=True)
    
    # Calculate the end voltage of the previous step
    df_statis['Last End Voltage(V)'] = df_statis['End Voltage(V)'].shift()

    df_statis['DCIR(Ohm)'] = 1000 * ((df_statis['Start Voltage(V)'] - df_statis['Last End Voltage(V)']) / df_statis['Start Current(mA)']).abs()
    
    return df_statis

#class for a button to select files
class SelectFilesButton(widgets.Button):
    """A file widget that leverages tkinter.filedialog."""

    def __init__(self):
        super(SelectFilesButton, self).__init__()
            self.add_traits(files=traitlets.traitlets.List())
            self.description = "Select Files"
        self.icon = "square-o"
        self.style.button_color = "orange"
            self.on_click(self.select_files)

    @staticmethod
    def select_files(b):
        """Generate instance of tkinter.filedialog.

        Parameters
        ----------
        b : obj:
            An instance of ipywidgets.widgets.Button 
        """
        with out:
            try:
                        root = Tk()
                        root.withdraw()
                        root.call('wm', 'attributes', '.', '-topmost', True)
                        b.files = filedialog.askopenfilename(multiple=True)

                b.description = "Files Selected"
                b.icon = "check-square-o"
                b.style.button_color = "lightgreen"
            except:
                pass

#class for button to select files
class SelectDirectoryButton(widgets.Button):
    """A file widget that leverages tkinter.filedialog."""

    def __init__(self):
        super(SelectDirectoryButton, self).__init__()
            self.add_traits(current_directory=traitlets.traitlets.Unicode())
            self.description = "Select Directory"
        self.icon = "square-o"
        self.style.button_color = "orange"
            self.on_click(self.select_directory)

    @staticmethod
    def select_directory(b):
        """Generate instance of tkinter.filedialog.

        Parameters
        ----------
        b : obj:
            An instance of ipywidgets.widgets.Button 
        """
        with out:
            try:
                        root = Tk()
                        root.withdraw()
                        root.call('wm', 'attributes', '.', '-topmost', True)
                        b.current_directory = filedialog.askdirectory()

                b.description = "Directory Selected"
                b.icon = "check-square-o"
                b.style.button_color = "lightgreen"
            except:
                pass
out = widgets.Output()
raw = SelectDirectoryButton()
widgets.VBox([raw, out])

# Optional preset for non-interactive runs
if PRESET_RAW_DIR is not None:
    raw.current_directory = PRESET_RAW_DIR

# Select Metadata file

In [ ]:
raw.current_directory
files = os.listdir(raw.current_directory)
met = widgets.Output()
metapath = SelectFilesButton()
widgets.VBox([metapath, met])

if PRESET_META_XLSX is not None:
    metapath.files = [PRESET_META_XLSX]

# Upload the Cycle Routine Mapping

In [ ]:
raw.current_directory
files = os.listdir(raw.current_directory)
cycle_map = widgets.Output()
cycle_path = SelectFilesButton()
widgets.VBox([cycle_path, cycle_map])

if PRESET_CYCLE_ROUTINE is not None:
    cycle_path.files = [PRESET_CYCLE_ROUTINE]

# Enter output file name

In [ ]:
metadata = pd.read_excel(metapath.files[0])
colors = ['#000000', '#008463', '#faaa00', '#006384', '#8A2DFB', '#640a32', '#ffd700', '#B2B2B2', '#FF2525', '#520DFF', '#00FF00', '#E49EDD', '#00FFFF', '#996633', '#666699', '#C4009A', '#000066', '#d4e157', '#83E28E', '#ff6f00', '#64ffda', '#f50057', '#e3f2fd', '#D6AD84', '#920095', '#808000', '#FFFF00', '#009900', '#3399FF', '#A52A2A', '#5F9EA0', '#FF69B4', '#BC8F8F', '#DFCAEE', '#769B63', '#CC9900'] + list(matplotlib.colors.cnames.values())
groups = metadata.Group.unique()
group_map ={}
color = 0
for i in groups:
    group_map[i]=colors[color]
    color+=1
metadata['Color'] = metadata.Group.map(group_map)

file_name = PRESET_OUTPUT_NAME if PRESET_OUTPUT_NAME is not None else input('Please input a file name for the output: ')
if file_name.endswith('.xlsx'):
    
    pass
# elif file_name.endswith('nda'):
#     file_name = file_name.replace('nda','.xlsx')
else:
    file_name = file_name+'.xlsx'
output = widgets.Output()
saveto = SelectDirectoryButton()
widgets.VBox([saveto, output])

if PRESET_OUTPUT_DIR is not None:
    saveto.current_directory = PRESET_OUTPUT_DIR

# Confirm output file location

In [ ]:
path=saveto.current_directory+'/'+file_name
print('Your output file will be saved to {}'.format(path))

# Establish DCIR steps

In [ ]:
#isolate soc positions
cycle_paths = pd.read_excel(cycle_path.files[0])
soc = cycle_paths[cycle_paths['State'].isin(['SOC 20%','SOC 50%','SOC Chg 50%','ignore'])]['NW cycles'].values[:].tolist()
all_soc = soc
socs = cycle_paths[cycle_paths['State'].isin(['SOC 20%','SOC 50%','SOC Chg 50%'])][['NW cycles','Correct cycles']].set_index('NW cycles').to_dict()['Correct cycles']
socs_states = cycle_paths[cycle_paths['State'].isin(['SOC 20%','SOC 50%','SOC Chg 50%'])][['NW cycles','State']].set_index('NW cycles').to_dict()['State']

# Enter your custom Average Charge Rate
Please input what percent SOC you want average charge rate (ACR) calculated. For example, enter 80 for 80%. If left empty, defaults to 80%. 

In [ ]:
acr_string = str(PRESET_CUSTOM_ACR) if PRESET_CUSTOM_ACR is not None else input('Custom ACR:')
acr_percent = float(acr_string)/100
if (acr_percent<1.0) & (acr_percent>0.0):
    pass
else:
    acr_percent = 0.8

# Process Data and Produce Charts

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm import tqdm


# function to prune the series so the legend has only 1 bar/group and not 1 line/cell
def prune_legend(metadata):
    positions = []
    index = pd.Index(metadata.Group)
    groups = metadata.Group.unique()
    for group in groups:
        position = index.get_loc(group)
        if isinstance(position, slice):
            positions.append(position.start)
        elif isinstance(position, np.ndarray):
            positions.append(np.where(position)[0].tolist()[0])
        else:
            positions.append(position)
    return list(set(range(len(index))) - set(positions))


def make_safe_sheet_name(name: str, max_length: int = 31) -> str:
    invalid_chars = r'[\\/*?:[\]]'
    safe_name = re.sub(invalid_chars, "_", name)
    return safe_name[:max_length] if len(safe_name) > max_length else safe_name


def make_safe_chart_sheet_name(name: str, max_length: int = 31) -> str:
    return make_safe_sheet_name(name, max_length)


def adjust_time(df, time_col, new_time_col):
    time_diff = np.diff(df[time_col], prepend=0)
    reset_idx = time_diff < 0
    if True in reset_idx:
        increments = df[time_col][reset_idx] - time_diff[reset_idx]
        true_indices = np.where(reset_idx)[0]
        new_array = np.zeros(reset_idx.shape, dtype=float)
        new_array[true_indices] = increments
        adjustment = np.cumsum(new_array)
        df[new_time_col] = df[time_col] + adjustment
    else:
        df[new_time_col] = df[time_col]
    return df


def as_float64_series(s):
    return pd.to_numeric(s, errors='coerce').astype('float64')


def safe_divide(numer, denom, fill_value=np.nan):
    """
    Return float64 numpy array from numer/denom with safe handling.
    """
    numer_arr = pd.to_numeric(numer, errors='coerce').to_numpy(dtype=np.float64)
    denom_arr = pd.to_numeric(denom, errors='coerce').to_numpy(dtype=np.float64)

    out = np.full(len(numer_arr), fill_value, dtype=np.float64)
    valid = np.isfinite(denom_arr) & (denom_arr != 0)
    np.divide(numer_arr, denom_arr, out=out, where=valid)
    return out


def add_1c_cycle_excl_first(cycle_map_df):
    """
    Create a new cycle column that excludes the first 1C point
    of each contiguous 1C block, while keeping a GLOBAL continuous numbering.
    """
    cm = cycle_map_df.copy()

    cm['Corrected Cycle'] = pd.to_numeric(cm['Corrected Cycle'], errors='coerce')
    cm['1C Cycle'] = pd.to_numeric(cm['1C Cycle'], errors='coerce')
    cm['NW Cycle'] = pd.to_numeric(cm['NW Cycle'], errors='coerce')

    cm = cm.sort_values(['Corrected Cycle', 'NW Cycle']).reset_index(drop=True)

    new_vals = [np.nan] * len(cm)
    valid_idx = cm.index[cm['1C Cycle'].notna()].tolist()

    if len(valid_idx) == 0:
        cm['1C Cycle Excl First'] = np.nan
        return cm

    global_counter = 0
    prev_corr = None

    for idx in valid_idx:
        curr_corr = cm.loc[idx, 'Corrected Cycle']
        is_new_block = (prev_corr is None) or (curr_corr != prev_corr + 1)

        if is_new_block:
            new_vals[idx] = np.nan
        else:
            global_counter += 1
            new_vals[idx] = global_counter

        prev_corr = curr_corr

    cm['1C Cycle Excl First'] = new_vals
    return cm


def load_cycle_map(cycle_map_path):
    cm = pd.read_excel(cycle_map_path).rename(columns={
        'NW cycles': 'NW Cycle',
        'Correct cycles': 'Corrected Cycle',
        '1C_cycles': '1C Cycle',
        'note': 'Note',
        'State': 'State'
    })
    cm = cm[['NW Cycle', 'Corrected Cycle', '1C Cycle', 'Note', 'State']].copy()

    for c in ['NW Cycle', 'Corrected Cycle', '1C Cycle']:
        cm[c] = pd.to_numeric(cm[c], errors='coerce')

    for c in ['Note', 'State']:
        cm[c] = cm[c].apply(lambda x: x.strip() if isinstance(x, str) else x)

    cm = cm.dropna(subset=['NW Cycle']).copy()
    cm = add_1c_cycle_excl_first(cm)
    return cm


def attach_cycle_map_info(df, cycle_map_df):
    df = df.copy()
    df['Cycle'] = pd.to_numeric(df['Cycle'], errors='coerce')
    cm = cycle_map_df.copy()
    cm['NW Cycle'] = pd.to_numeric(cm['NW Cycle'], errors='coerce')
    cm_map = cm.drop_duplicates(subset=['NW Cycle']).copy()

    df['1C Cycle'] = df['Cycle'].map(dict(zip(cm_map['NW Cycle'], cm_map['1C Cycle'])))
    df['1C Cycle Excl First'] = df['Cycle'].map(dict(zip(cm_map['NW Cycle'], cm_map['1C Cycle Excl First'])))
    df['Cycle Note'] = df['Cycle'].map(dict(zip(cm_map['NW Cycle'], cm_map['Note'])))
    df['Cycle State'] = df['Cycle'].map(dict(zip(cm_map['NW Cycle'], cm_map['State'])))
    df['Map Corrected Cycle'] = df['Cycle'].map(dict(zip(cm_map['NW Cycle'], cm_map['Corrected Cycle'])))
    return df


def col_idx(df, col_name):
    return df.columns.get_loc(col_name)


def add_series_by_col(chart, sheet_name, nrows, x_col, y_col, color, name):
    if nrows <= 0:
        return
    chart.add_series({
        'categories': [sheet_name, 1, x_col, nrows, x_col],
        'values': [sheet_name, 1, y_col, nrows, y_col],
        'line': {'color': color},
        'name': name,
    })


def clean_series(s):
    return pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()


def update_range(ranges, key, s):
    s = clean_series(s)
    if len(s) == 0:
        return
    ranges[key]['min'] = min(ranges[key]['min'], s.min())
    ranges[key]['max'] = max(ranges[key]['max'], s.max())


def update_max(max_tracker, key, value):
    if pd.notna(value):
        max_tracker[key] = float(max(max_tracker.get(key, 0), value))

def add_df_series(chart, df, sheet_name, x_col_name, y_col_name, color, name):
    if len(df) == 0 or x_col_name not in df.columns or y_col_name not in df.columns:
        return
    add_series_by_col(
        chart, sheet_name, len(df),
        col_idx(df, x_col_name),
        col_idx(df, y_col_name),
        color, name
    )


def set_chart_axis(chart, x_name, y_name, x_min, x_max, y_min, y_max, remove_legends):
    chart.set_x_axis({
        'name': x_name,
        'name_font': {'name': 'Helvetica Neue', 'size': 14, 'bold': True},
        'min': x_min,
        'max': x_max,
    })
    chart.set_y_axis({
        'name': y_name,
        'name_font': {'name': 'Helvetica Neue', 'size': 14, 'bold': True},
        'min': y_min,
        'max': y_max,
    })
    chart.set_legend({'delete_series': remove_legends})


def build_dcir_subset(df_dcir, include_dcir0=False):
    if len(df_dcir) == 0:
        return None
    out = df_dcir.copy()
    out['nDCIR'] = out['DCIR'] / out['DCIR'].iloc[0]
    if include_dcir0 and 'DCIR0' in out.columns and 'DCIRt' in out.columns:
        out['nDCIR0'] = out['DCIR0'] / out['DCIR0'].iloc[0]
        out['nDCIRt'] = out['DCIRt'] / out['DCIRt'].iloc[0]
    return out


# =========================
# 1C feature extraction helpers
# =========================
def get_exact_cycle_value(s, cycle_target, cycle_col='1C Cycle Excl First'):
    temp = s.loc[s[cycle_col] == cycle_target, 'y']
    if len(temp) == 0:
        return np.nan
    val = pd.to_numeric(temp, errors='coerce').iloc[0]
    return val if pd.notna(val) else np.nan


def get_end_minus_one_value(s, cycle_col='1C Cycle Excl First'):
    if len(s) == 0:
        return np.nan
    max_cycle = s[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan
    target = max_cycle - 1
    if target < 1:
        return np.nan
    val = get_exact_cycle_value(s, target, cycle_col=cycle_col)
    if pd.isna(val):
        return get_exact_cycle_value(s, max_cycle, cycle_col=cycle_col)
    return val


def calc_slope_between_points(s, cycle_start, cycle_end, cycle_col='1C Cycle Excl First'):
    if len(s) == 0:
        return np.nan

    max_cycle = s[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan

    cycle_end_safe = min(cycle_end, max_cycle)
    if cycle_end_safe <= cycle_start:
        return np.nan

    y1 = get_exact_cycle_value(s, cycle_start, cycle_col=cycle_col)
    y2 = get_exact_cycle_value(s, cycle_end_safe, cycle_col=cycle_col)

    if pd.isna(y1) or pd.isna(y2):
        return np.nan

    return (y2 - y1) / (cycle_end_safe - cycle_start)


def calc_slope_1_to_endm1(s, cycle_col='1C Cycle Excl First'):
    if len(s) == 0:
        return np.nan

    max_cycle = s[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan

    endm1 = max_cycle - 1
    if endm1 <= 1:
        endm1 = max_cycle

    if endm1 <= 1:
        return np.nan

    return calc_slope_between_points(s, 1, endm1, cycle_col=cycle_col)


def calc_abruptness(s, cycle_col='1C Cycle Excl First'):
    if len(s) == 0:
        return np.nan

    s2 = s.copy()
    s2 = s2[(s2[cycle_col] >= 1)].copy()
    if len(s2) < 2:
        return np.nan

    max_cycle = s2[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan

    endm1 = max_cycle - 1
    if endm1 < 1:
        endm1 = max_cycle

    s2 = s2[s2[cycle_col] <= endm1].copy()
    if len(s2) == 0:
        return np.nan

    f1 = get_exact_cycle_value(s2, 1, cycle_col=cycle_col)
    fend = get_exact_cycle_value(s2, endm1, cycle_col=cycle_col)

    if pd.isna(f1) or pd.isna(fend):
        return np.nan

    denom = f1 - fend
    if pd.isna(denom) or denom == 0:
        return np.nan

    z = (pd.to_numeric(s2['y'], errors='coerce') - fend) / denom
    z = z.replace([np.inf, -np.inf], np.nan).dropna()

    if len(z) == 0:
        return np.nan

    return 1 - 2 * z.mean()


def get_final_1c_cycle_number(df_in, cycle_col='1C Cycle'):
    if cycle_col not in df_in.columns or len(df_in) == 0:
        return np.nan
    s = pd.to_numeric(df_in[cycle_col], errors='coerce').dropna()
    if len(s) == 0:
        return np.nan
    return s.max()


def get_end_minus_n_value(s, n, cycle_col='1C Cycle Excl First'):
    if len(s) == 0:
        return np.nan

    max_cycle = s[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan

    target = max_cycle - n
    if target < 1:
        return np.nan

    return get_exact_cycle_value(s, target, cycle_col=cycle_col)


def calc_slope_if_valid(s, cycle_start, cycle_end, cycle_col='1C Cycle Excl First', require_max_cycle_at_least=None):
    if len(s) == 0:
        return np.nan

    max_cycle = s[cycle_col].max()
    if pd.isna(max_cycle):
        return np.nan

    if require_max_cycle_at_least is not None and max_cycle < require_max_cycle_at_least:
        return np.nan

    if cycle_start < 1 or cycle_end < 1:
        return np.nan

    return calc_slope_between_points(s, cycle_start, cycle_end, cycle_col=cycle_col)


def extract_1c_features_extended_from_df(df_in, y_col, cycle_col='1C Cycle Excl First'):
    default_result = {
        'f_1cyc': np.nan,
        'f_50cyc': np.nan,
        'f_endm50cyc': np.nan,
        'f_endm1cyc': np.nan,
        'df_dcyc_1_50': np.nan,
        'df_dcyc_50_endm50': np.nan,
        'df_dcyc_endm50_endm1': np.nan,
        'df_dcyc_1_endm1': np.nan,
        'abruptness': np.nan,
    }

    if y_col not in df_in.columns or cycle_col not in df_in.columns:
        return default_result

    s = df_in[[cycle_col, y_col]].copy()
    s.columns = [cycle_col, 'y']
    s[cycle_col] = pd.to_numeric(s[cycle_col], errors='coerce')
    s['y'] = pd.to_numeric(s['y'], errors='coerce')
    s = s.dropna(subset=[cycle_col, 'y']).sort_values(cycle_col).drop_duplicates(subset=[cycle_col])

    if len(s) == 0:
        return default_result

    max_cycle = s[cycle_col].max()

    if pd.notna(max_cycle) and max_cycle >= 100:
        f_endm50 = get_end_minus_n_value(s, 50, cycle_col=cycle_col)
        slope_50_to_endm50 = calc_slope_if_valid(
            s, 50, max_cycle - 50,
            cycle_col=cycle_col,
            require_max_cycle_at_least=100
        )
        slope_endm50_to_endm1 = calc_slope_if_valid(
            s, max_cycle - 50, max_cycle - 1,
            cycle_col=cycle_col,
            require_max_cycle_at_least=100
        )
    else:
        f_endm50 = np.nan
        slope_50_to_endm50 = np.nan
        slope_endm50_to_endm1 = np.nan

    return {
        'f_1cyc': get_exact_cycle_value(s, 1, cycle_col=cycle_col),
        'f_50cyc': get_exact_cycle_value(s, 50, cycle_col=cycle_col),
        'f_endm50cyc': f_endm50,
        'f_endm1cyc': get_end_minus_one_value(s, cycle_col=cycle_col),
        'df_dcyc_1_50': calc_slope_between_points(s, 1, 50, cycle_col=cycle_col),
        'df_dcyc_50_endm50': slope_50_to_endm50,
        'df_dcyc_endm50_endm1': slope_endm50_to_endm1,
        'df_dcyc_1_endm1': calc_slope_1_to_endm1(s, cycle_col=cycle_col),
        'abruptness': calc_abruptness(s, cycle_col=cycle_col),
    }


def average_between_cycles(df_in, y_col, start_cycle, end_cycle, cycle_col='1C Cycle Excl First'):
    if y_col not in df_in.columns or cycle_col not in df_in.columns or len(df_in) == 0:
        return np.nan

    s = df_in[[cycle_col, y_col]].copy()
    s[cycle_col] = pd.to_numeric(s[cycle_col], errors='coerce')
    s[y_col] = pd.to_numeric(s[y_col], errors='coerce')
    s = s.dropna(subset=[cycle_col, y_col])

    s = s[(s[cycle_col] >= start_cycle) & (s[cycle_col] <= end_cycle)].copy()
    if len(s) == 0:
        return np.nan

    return s[y_col].mean()


def extract_ce_like_features(df_in, y_col, cycle_col='1C Cycle Excl First'):
    default_result = {
        'f_1cyc': np.nan,
        'avg_1_25': np.nan,
        'avg_25_100': np.nan,
        'avg_100_endm50': np.nan,
        'avg_endm50_endm1': np.nan,
    }

    if y_col not in df_in.columns or cycle_col not in df_in.columns:
        return default_result

    s = df_in[[cycle_col, y_col]].copy()
    s[cycle_col] = pd.to_numeric(s[cycle_col], errors='coerce')
    s[y_col] = pd.to_numeric(s[y_col], errors='coerce')
    s = s.dropna(subset=[cycle_col, y_col]).sort_values(cycle_col).drop_duplicates(subset=[cycle_col])

    if len(s) == 0:
        return default_result

    max_cycle = s[cycle_col].max()

    result = {
        'f_1cyc': get_exact_cycle_value(
            s.rename(columns={y_col: 'y'}),
            1,
            cycle_col=cycle_col
        ),
        'avg_1_25': average_between_cycles(s, y_col, 1, 25, cycle_col=cycle_col),
        'avg_25_100': average_between_cycles(s, y_col, 25, 100, cycle_col=cycle_col),
        'avg_100_endm50': np.nan,
        'avg_endm50_endm1': np.nan,
    }

    if pd.notna(max_cycle) and max_cycle >= 100:
        endm50 = max_cycle - 50
        endm1 = max_cycle - 1

        if endm50 >= 100:
            result['avg_100_endm50'] = average_between_cycles(s, y_col, 100, endm50, cycle_col=cycle_col)

        if endm1 >= endm50 and endm50 >= 1:
            result['avg_endm50_endm1'] = average_between_cycles(s, y_col, endm50, endm1, cycle_col=cycle_col)

    return result


# =========================
# C10 summary helpers
# =========================
def safe_pct_delta(v1, v2):
    if pd.isna(v1) or pd.isna(v2) or v1 == 0:
        return np.nan
    return (v2 - v1) / v1 * 100


def safe_rate_per_cycle(v1, v2, cyc1, cyc2):
    if pd.isna(v1) or pd.isna(v2) or pd.isna(cyc1) or pd.isna(cyc2):
        return np.nan
    if cyc2 <= cyc1:
        return np.nan
    if v1 == 0:
        return np.nan
    return ((v2 - v1) / v1 * 100) / (cyc2 - cyc1)


def build_c10_summary_row(
    cell,
    electrolyte_name,
    df_c10,
    df_kinetic,
    soc50_df
):
    row = {
        'Barcode': cell,
        'Electrolyte': electrolyte_name,
    }

    c10_cols = [
        'Charge Capacity',
        'Discharge Capacity',
        'Charge Energy (mWh)',
        'Discharge Energy (mWh)',
        'Average Charge Voltage',
        'Average Discharge Voltage',
        'Coulombic Efficiency (%)',
        'RCE',
        'Charge Resistance',
        'Discharge Resistance',
        '100% ACR',
        'Energy Efficiency %',
        'Voltage Slippage',
        'V Polarization',
    ]

    soc50_cols = ['DCIR', 'DCIR0', 'DCIRt']

    idx_map = {
        'f_1': 0,
        'f_2': 1,
        'f_end': -1,
    }

    c10_snapshots = {}
    soc50_snapshots = {}

    for snap_name, idx in idx_map.items():
        # C10
        if df_c10 is not None and len(df_c10) > 0:
            real_idx = idx if idx >= 0 else len(df_c10) - 1
            if 0 <= real_idx < len(df_c10):
                c10_row = df_c10.iloc[real_idx]
                c10_snapshots[snap_name] = c10_row

                for c in c10_cols:
                    row[f'{snap_name} | C10 | {c}'] = c10_row[c] if c in df_c10.columns else np.nan
            else:
                c10_snapshots[snap_name] = None
                for c in c10_cols:
                    row[f'{snap_name} | C10 | {c}'] = np.nan
        else:
            c10_snapshots[snap_name] = None
            for c in c10_cols:
                row[f'{snap_name} | C10 | {c}'] = np.nan

        # SOC50
        if soc50_df is not None and len(soc50_df) > 0:
            real_idx = idx if idx >= 0 else len(soc50_df) - 1
            if 0 <= real_idx < len(soc50_df):
                soc50_row = soc50_df.iloc[real_idx]
                soc50_snapshots[snap_name] = soc50_row
                for c in soc50_cols:
                    row[f'{snap_name} | SOC50 | {c}'] = soc50_row[c] if c in soc50_df.columns else np.nan
            else:
                soc50_snapshots[snap_name] = None
                for c in soc50_cols:
                    row[f'{snap_name} | SOC50 | {c}'] = np.nan
        else:
            soc50_snapshots[snap_name] = None
            for c in soc50_cols:
                row[f'{snap_name} | SOC50 | {c}'] = np.nan

    # only keep end cycle info
    if c10_snapshots['f_end'] is not None and 'Map Corrected Cycle' in c10_snapshots['f_end'].index:
        row['f_end | C10 | Map Corrected Cycle'] = c10_snapshots['f_end']['Map Corrected Cycle']
    else:
        row['f_end | C10 | Map Corrected Cycle'] = np.nan

    # delta blocks
    kinetic_1 = df_kinetic.iloc[0] if df_kinetic is not None and len(df_kinetic) >= 1 else None
    kinetic_end = df_kinetic.iloc[-1] if df_kinetic is not None and len(df_kinetic) >= 1 else None

    row['delta_1_to_2 | Kinetic Fading (%)'] = kinetic_1['Kinetic Fading (%)'] if kinetic_1 is not None and 'Kinetic Fading (%)' in kinetic_1.index else np.nan
    row['delta_1_to_end | Kinetic Fading (%)'] = kinetic_end['Kinetic Fading (%)'] if kinetic_end is not None and 'Kinetic Fading (%)' in kinetic_end.index else np.nan

    c10_f1 = c10_snapshots['f_1']
    c10_f2 = c10_snapshots['f_2']
    c10_fend = c10_snapshots['f_end']

    cyc1 = c10_f1['Map Corrected Cycle'] if c10_f1 is not None and 'Map Corrected Cycle' in c10_f1.index else np.nan
    cyc2 = c10_f2['Map Corrected Cycle'] if c10_f2 is not None and 'Map Corrected Cycle' in c10_f2.index else np.nan
    cycend = c10_fend['Map Corrected Cycle'] if c10_fend is not None and 'Map Corrected Cycle' in c10_fend.index else np.nan

    for c in c10_cols:
        v1 = c10_f1[c] if c10_f1 is not None and c in c10_f1.index else np.nan
        v2 = c10_f2[c] if c10_f2 is not None and c in c10_f2.index else np.nan
        vend = c10_fend[c] if c10_fend is not None and c in c10_fend.index else np.nan

        row[f'delta_1_to_2 | C10 | {c} | pct'] = safe_pct_delta(v1, v2)
        row[f'delta_1_to_2 | C10 | {c} | pct_per_cyc'] = safe_rate_per_cycle(v1, v2, cyc1, cyc2)

        row[f'delta_1_to_end | C10 | {c} | pct'] = safe_pct_delta(v1, vend)
        row[f'delta_1_to_end | C10 | {c} | pct_per_cyc'] = safe_rate_per_cycle(v1, vend, cyc1, cycend)

    return row


def build_full_cell_table(df, total_cap, acr_string):
    out = df.copy()
    extra_cols = [
        'Cycle',
        'Total Charge Capacity(mAh)',
        'Total Step time',
        f'{acr_string}% ACR',
        f'{acr_string}% Time',
        '20% Time',
        '20% Time Ratio'
    ]
    extra_cols = [c for c in extra_cols if c in total_cap.columns]
    extra = total_cap[extra_cols].drop_duplicates(subset=['Cycle']).copy()
    out = out.merge(extra, on='Cycle', how='left')
    return out


def build_1c_ex1_table(df_1c_ex1, total_cap_1c_ex1, acr_string):
    out = df_1c_ex1.copy()
    extra_cols = [
        'Cycle',
        '1C Cycle Excl First',
        'Total Charge Capacity(mAh)',
        'Total Step time',
        f'{acr_string}% ACR',
        f'{acr_string}% Time',
        '20% Time',
        '20% Time Ratio'
    ]
    extra_cols = [c for c in extra_cols if c in total_cap_1c_ex1.columns]
    extra = total_cap_1c_ex1[extra_cols].drop_duplicates(subset=['Cycle', '1C Cycle Excl First']).copy()
    out = out.merge(extra, on=['Cycle', '1C Cycle Excl First'], how='left')
    return out


def build_c10_combined_table(df_c10, df_kinetic, soc50_df):
    pieces = []

    if df_c10 is not None and len(df_c10) > 0:
        temp = df_c10.copy().reset_index(drop=True)
        temp.insert(0, 'Row ID', np.arange(1, len(temp) + 1))
        temp = temp.rename(columns={'Map Corrected Cycle': 'C10 | Map Corrected Cycle'})
        temp = temp.rename(columns={c: f'C10 | {c}' for c in temp.columns if c not in ['Row ID', 'C10 | Map Corrected Cycle']})
        pieces.append(temp)

    if df_kinetic is not None and len(df_kinetic) > 0:
        temp = df_kinetic.copy().reset_index(drop=True)
        temp.insert(0, 'Row ID', np.arange(1, len(temp) + 1))
        temp = temp.rename(columns={'End C/10 Map Cycle': 'Kinetic | End C/10 Map Cycle'})
        temp = temp.rename(columns={c: f'Kinetic | {c}' for c in temp.columns if c not in ['Row ID', 'Kinetic | End C/10 Map Cycle']})
        pieces.append(temp)

    if soc50_df is not None and len(soc50_df) > 0:
        temp = soc50_df.copy().reset_index(drop=True)
        temp.insert(0, 'Row ID', np.arange(1, len(temp) + 1))
        temp = temp.rename(columns={'Corrected Cycle': 'SOC50 | DCIR Cycle'})
        temp = temp.rename(columns={c: f'SOC50 | {c}' for c in temp.columns if c not in ['Row ID', 'SOC50 | DCIR Cycle']})
        pieces.append(temp)

    if len(pieces) == 0:
        return pd.DataFrame()

    out = pieces[0].copy()
    for p in pieces[1:]:
        out = out.merge(p, on='Row ID', how='outer')

    return out


# -------------------------
# Load cycle map file
# -------------------------
cycle_map_file = cycle_path.files[0]
cycle_map_df = load_cycle_map(cycle_map_file)

with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
    ranges = {
        'soc50_dcir': {'min': 10, 'max': 0},
        'soc50_ndcir': {'min': 10, 'max': 0},
        'soc50_dcir0': {'min': 10, 'max': 0},
        'soc50_dcirt': {'min': 10, 'max': 0},
        'soc50_ndcir0': {'min': 10, 'max': 0},
        'soc50_ndcirt': {'min': 10, 'max': 0},
        'charge': {'min': 100000, 'max': 0},
        'discharge': {'min': 100000, 'max': 0},
        'charge_energy': {'min': 100000, 'max': 0},
        'discharge_energy': {'min': 100000, 'max': 0},
        'charge_energy_retention': {'min': 100000, 'max': 0},
        'discharge_energy_retention': {'min': 100000, 'max': 0},
        'avg_charge': {'min': 10, 'max': 0},
        'avg_discharge': {'min': 10, 'max': 0},
        'ce': {'min': 150, 'max': 0},
        'rce': {'min': 300, 'max': 0},
        'cum_ce_loss': {'min': 0, 'max': 0},
        'cum_rce_loss': {'min': 0, 'max': 0},
        'hi1': {'min': 10, 'max': 0},
        'res_chg': {'min': 1000000, 'max': 0},
        'res_dchg': {'min': 1000000, 'max': 0},
        'nres_chg': {'min': 1000000, 'max': 0},
        'nres_dchg': {'min': 1000000, 'max': 0},
        'acr': {'min': 100000, 'max': 0},
        'custom_acr': {'min': 100000, 'max': 0},
        'dv_chg': {'min': 100000, 'max': 0},
        'ndv_chg': {'min': 100000, 'max': 0},
        'dv_dchg': {'min': 100000, 'max': 0},
        'ndv_dchg': {'min': 100000, 'max': 0},
        'energy_eff': {'min': 100000, 'max': 0},
        'ratio20': {'min': 1e9, 'max': 0},
        'vslip': {'min': 10, 'max': 0},
        'vpol': {'min': 10, 'max': 0},
        'c10_retention': {'min': 100000, 'max': 0},
        'kinetic_fade': {'min': 100000, 'max': 0},
    }

    max_tracker = {
        'cycle': 0,
        'cycle_1c_ex1': 0,
        'soc50_cycle': 0,
        'cycle_life': 0,
        'c10_cycle': 0,
        'kinetic_cycle': 0,
    }

    exists = {'soc50': False}
    chart_has_series_c10 = False
    chart_has_series_kinetic = False

    metric_map = {
        'Charge Capacity': 'Charge Capacity',
        'Discharge Capacity': 'Discharge Capacity',
        'Capacity Retention (%)': 'Cycle Life',
        'Coulombic Efficiency (%)': 'Coulombic Efficiency (%)',
        'Charge Energy (mWh)': 'Charge Energy (mWh)',
        'Discharge Energy (mWh)': 'Discharge Energy (mWh)',
        'Energy Efficiency (%)': 'Energy Efficiency %',
        'Avg Charge Voltage (V)': 'Average Charge Voltage',
        'Avg Discharge Voltage (V)': 'Average Discharge Voltage',
        'Voltage Slippage (V)': 'Voltage Slippage',
        'V Polarization (V)': 'V Polarization',
        'Kinetic Fading (%)': 'Kinetic Fading (%)',
    }

    plot_1c_specs = {
        'charge_capacity_ex1': {
            'df_col': 'Charge Capacity',
            'title': 'Charge Capacity 1C Excl First',
            'y_name': 'Charge Capacity (mAh)',
            'source': 'df',
            'range_key': 'charge'
        },
        'discharge_capacity_ex1': {
            'df_col': 'Discharge Capacity',
            'title': 'Discharge Capacity 1C Excl First',
            'y_name': 'Discharge Capacity (mAh)',
            'source': 'df',
            'range_key': 'discharge'
        },
        'cycle_life_ex1': {
            'df_col': 'Cycle Life 1C',
            'title': 'Cycle Life 1C Excl First',
            'y_name': 'Cycle Life (%)',
            'source': 'df',
            'range_key': 'cycle_life'
        },
        'charge_energy_ex1': {
            'df_col': 'Charge Energy (mWh)',
            'title': 'Charge Energy 1C Excl First',
            'y_name': 'Charge Energy (mWh)',
            'source': 'df',
            'range_key': 'charge_energy'
        },
        'discharge_energy_ex1': {
            'df_col': 'Discharge Energy (mWh)',
            'title': 'Discharge Energy 1C Excl First',
            'y_name': 'Discharge Energy (mWh)',
            'source': 'df',
            'range_key': 'discharge_energy'
        },
        'charge_energy_retention_ex1': {
            'df_col': 'Charge Energy Retention 1C',
            'title': 'Charge Energy Retention 1C Excl First',
            'y_name': 'Charge Energy Retention',
            'source': 'df',
            'range_key': 'charge_energy_retention'
        },
        'discharge_energy_retention_ex1': {
            'df_col': 'Discharge Energy Retention 1C',
            'title': 'Discharge Energy Retention 1C Excl First',
            'y_name': 'Discharge Energy Retention',
            'source': 'df',
            'range_key': 'discharge_energy_retention'
        },
        'avg_charge_voltage_ex1': {
            'df_col': 'Average Charge Voltage',
            'title': 'Avg Charge Voltage 1C Excl First',
            'y_name': 'Average Charge Voltage (V)',
            'source': 'df',
            'range_key': 'avg_charge'
        },
        'avg_discharge_voltage_ex1': {
            'df_col': 'Average Discharge Voltage',
            'title': 'Avg Discharge Voltage 1C Excl First',
            'y_name': 'Average Discharge Voltage (V)',
            'source': 'df',
            'range_key': 'avg_discharge'
        },
        'ce_ex1': {
            'df_col': 'Coulombic Efficiency (%)',
            'title': 'CE 1C Excl First',
            'y_name': 'Coulombic Efficiency (%)',
            'source': 'df',
            'range_key': 'ce'
        },
        'rce_ex1': {
            'df_col': 'RCE',
            'title': 'RCE 1C Excl First',
            'y_name': 'RCE (%)',
            'source': 'df',
            'range_key': 'rce'
        },
        'cum_ce_loss_ex1': {
            'df_col': 'Cumulative CE Loss',
            'title': 'Cumulative CE Loss 1C Excl First',
            'y_name': 'Cumulative CE Loss (%)',
            'source': 'df',
            'range_key': 'cum_ce_loss'
        },
        'cum_rce_loss_ex1': {
            'df_col': 'Cumulative RCE Loss',
            'title': 'Cumulative RCE Loss 1C Excl First',
            'y_name': 'Cumulative RCE Loss (%)',
            'source': 'df',
            'range_key': 'cum_rce_loss'
        },
        'hi1_ex1': {
            'df_col': 'HI1',
            'title': 'HI1 1C Excl First',
            'y_name': 'HI1',
            'source': 'df',
            'range_key': 'hi1'
        },
        'charge_resistance_ex1': {
            'df_col': 'Charge Resistance',
            'title': 'Charge Resistance 1C Excl First',
            'y_name': 'Charge Resistance (Ohm)',
            'source': 'df',
            'range_key': 'res_chg'
        },
        'discharge_resistance_ex1': {
            'df_col': 'Discharge Resistance',
            'title': 'Discharge Resistance 1C Excl First',
            'y_name': 'Discharge Resistance (Ohm)',
            'source': 'df',
            'range_key': 'res_dchg'
        },
        'acr_100_ex1': {
            'df_col': '100% ACR',
            'title': '100% ACR 1C Excl First',
            'y_name': '100% Average Charge Rate',
            'source': 'df',
            'range_key': 'acr'
        },
        'acr_custom_ex1': {
            'df_col': f'{acr_string}% ACR',
            'title': f'{acr_string}% ACR 1C Excl First',
            'y_name': f'{acr_string}% Average Charge Rate',
            'source': 'acr',
            'range_key': 'custom_acr'
        },
        'energy_efficiency_ex1': {
            'df_col': 'Energy Efficiency %',
            'title': 'Energy Efficiency 1C Excl First',
            'y_name': 'Energy Efficiency %',
            'source': 'df',
            'range_key': 'energy_eff'
        },
        'time_ratio_20_ex1': {
            'df_col': '20% Time Ratio',
            'title': '20% Time Ratio 1C Excl First',
            'y_name': '20% Time / Total Time',
            'source': 'acr',
            'range_key': 'ratio20'
        },
        'v_slippage_ex1': {
            'df_col': 'Voltage Slippage',
            'title': 'Voltage Slippage 1C Excl First',
            'y_name': 'Voltage Slippage (V)',
            'source': 'df',
            'range_key': 'vslip'
        },
        'v_polarization_ex1': {
            'df_col': 'V Polarization',
            'title': 'V Polarization 1C Excl First',
            'y_name': 'V Polarization (V)',
            'source': 'df',
            'range_key': 'vpol'
        },
    }

    summary_1c_metrics = {
        'charge_capacity_ex1': {'source': 'df_1c_ex1', 'col': 'Charge Capacity'},
        'discharge_capacity_ex1': {'source': 'df_1c_ex1', 'col': 'Discharge Capacity'},
        'cycle_life_ex1': {'source': 'df_1c_ex1', 'col': 'Cycle Life 1C'},
        'avg_charge_voltage_ex1': {'source': 'df_1c_ex1', 'col': 'Average Charge Voltage'},
        'avg_discharge_voltage_ex1': {'source': 'df_1c_ex1', 'col': 'Average Discharge Voltage'},
        'acr_100_ex1': {'source': 'df_1c_ex1', 'col': '100% ACR'},
        'acr_custom_ex1': {'source': 'total_cap_1c_ex1', 'col': f'{acr_string}% ACR'},
        'energy_efficiency_ex1': {'source': 'df_1c_ex1', 'col': 'Energy Efficiency %'},
        'time_ratio_20_ex1': {'source': 'total_cap_1c_ex1', 'col': '20% Time Ratio'},
        'v_slippage_ex1': {'source': 'df_1c_ex1', 'col': 'Voltage Slippage'},
        'v_polarization_ex1': {'source': 'df_1c_ex1', 'col': 'V Polarization'},
        'cum_ce_loss_ex1': {'source': 'df_1c_ex1', 'col': 'Cumulative CE Loss'},
        'cum_rce_loss_ex1': {'source': 'df_1c_ex1', 'col': 'Cumulative RCE Loss'},
    }

    summary_ce_like_metrics = {
        'ce_ex1': {'source': 'df_1c_ex1', 'col': 'Coulombic Efficiency (%)'},
        'rce_ex1': {'source': 'df_1c_ex1', 'col': 'RCE'},
    }

    remove_legends = prune_legend(metadata)

    workbook = writer.book
    chartsheets = {}
    charts = {}

    def ensure_chart(sheet_name):
        if sheet_name not in charts:
            charts[sheet_name] = workbook.add_chart({'type': 'scatter', 'subtype': 'straight'})
        if sheet_name not in chartsheets:
            chartsheets[sheet_name] = workbook.add_chartsheet(make_safe_chart_sheet_name(sheet_name))

    main_specs = {
        'charge': {'sheet': 'Charge Capacity', 'y_col': 'Charge Capacity', 'y_name': 'Charge Capacity(mAh)', 'range_key': 'charge'},
        'discharge': {'sheet': 'Discharge Capacity', 'y_col': 'Discharge Capacity', 'y_name': 'Discharge Capacity(mAh)', 'range_key': 'discharge'},
        'charge_energy': {'sheet': 'Charge Energy (mWh)', 'y_col': 'Charge Energy (mWh)', 'y_name': 'Charge Energy (mWh)', 'range_key': 'charge_energy'},
        'discharge_energy': {'sheet': 'Discharge Energy (mWh)', 'y_col': 'Discharge Energy (mWh)', 'y_name': 'Discharge Energy (mWh)', 'range_key': 'discharge_energy'},
        'charge_energy_retention': {'sheet': 'Charge Energy Retention', 'y_col': 'Charge Energy Retention', 'y_name': 'Charge Energy Retention', 'range_key': 'charge_energy_retention'},
        'discharge_energy_retention': {'sheet': 'Discharge Energy Retention', 'y_col': 'Discharge Energy Retention', 'y_name': 'Discharge Energy Retention', 'range_key': 'discharge_energy_retention'},
        'avg_charge': {'sheet': 'Avg Charge Voltage', 'y_col': 'Average Charge Voltage', 'y_name': 'Average Charge Voltage(V)', 'range_key': 'avg_charge'},
        'avg_discharge': {'sheet': 'Avg Discharge Voltage', 'y_col': 'Average Discharge Voltage', 'y_name': 'Average Discharge Voltage(V)', 'range_key': 'avg_discharge'},
        'cycle_life': {'sheet': 'Cycle Life', 'y_col': 'Cycle Life', 'y_name': 'Cycle Life(%)', 'range_key': 'cycle_life'},
        'ce': {'sheet': 'CE', 'y_col': 'Coulombic Efficiency (%)', 'y_name': 'Coulombic Efficiency(%)', 'range_key': 'ce'},
        'rce': {'sheet': 'RCE', 'y_col': 'RCE', 'y_name': 'RCE(%)', 'range_key': 'rce'},
        'hi1': {'sheet': 'HI1', 'y_col': 'HI1', 'y_name': 'HI1', 'range_key': 'hi1'},
        'res_chg': {'sheet': 'Charge Resistance', 'y_col': 'Charge Resistance', 'y_name': 'Charge Resistance (Ohm)', 'range_key': 'res_chg'},
        'res_dchg': {'sheet': 'Discharge Resistance', 'y_col': 'Discharge Resistance', 'y_name': 'Discharge Resistance (Ohm)', 'range_key': 'res_dchg'},
        'nres_chg': {'sheet': 'nCharge Resistance', 'y_col': 'nCharge Resistance', 'y_name': 'nCharge Resistance', 'range_key': 'nres_chg'},
        'nres_dchg': {'sheet': 'nDischarge Resistance', 'y_col': 'nDischarge Resistance', 'y_name': 'nDischarge Resistance', 'range_key': 'nres_dchg'},
        'acr': {'sheet': '100% ACR', 'y_col': '100% ACR', 'y_name': '100% Average Charge Rate', 'range_key': 'acr'},
        'custom_acr': {'sheet': f'{acr_string}% ACR', 'y_col': f'{acr_string}% ACR', 'y_name': f'{acr_string}% Average Charge Rate', 'range_key': 'custom_acr', 'source': 'total_cap'},
        'dv_chg': {'sheet': 'Delta V Charge', 'y_col': 'Delta V Charge', 'y_name': 'Delta V Charge', 'range_key': 'dv_chg'},
        'dv_dchg': {'sheet': 'Delta V Discharge', 'y_col': 'Delta V Discharge', 'y_name': 'Delta V Discharge', 'range_key': 'dv_dchg'},
        'ndv_chg': {'sheet': 'nDelta V Charge', 'y_col': 'nDelta V Charge', 'y_name': 'nDelta V Charge', 'range_key': 'ndv_chg'},
        'ndv_dchg': {'sheet': 'nDelta V Discharge', 'y_col': 'nDelta V Discharge', 'y_name': 'nDelta V Discharge', 'range_key': 'ndv_dchg'},
        'energy_eff': {'sheet': 'Energy Efficiency (%)', 'y_col': 'Energy Efficiency %', 'y_name': 'Energy Efficiency %', 'range_key': 'energy_eff'},
        'ratio20': {'sheet': '20% Time Ratio', 'y_col': '20% Time Ratio', 'y_name': '20% Time / Total Time', 'range_key': 'ratio20', 'source': 'total_cap'},
        'vslip': {'sheet': 'Voltage Slippage', 'y_col': 'Voltage Slippage', 'y_name': 'Voltage Slippage (V)', 'range_key': 'vslip'},
        'vpol': {'sheet': 'V Polarization', 'y_col': 'V Polarization', 'y_name': 'V Polarization (V)', 'range_key': 'vpol'},
        'c10_retention': {'sheet': 'C10 Capacity Retention', 'y_col': 'C/10 Capacity Retention (%)', 'y_name': 'C/10 Capacity Retention (%)', 'range_key': 'c10_retention', 'source': 'c10'},
        'kinetic_fade': {'sheet': 'Kinetic Fading', 'y_col': 'Kinetic Fading (%)', 'y_name': 'Kinetic Fading (%)', 'range_key': 'kinetic_fade', 'source': 'kinetic'},
    }

    soc_specs = {
        'soc50_dcir': {'sheet': '50% SOC DCIR', 'y_col': 'DCIR', 'y_name': 'SOC 50% DCIR', 'range_key': 'soc50_dcir', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
        'soc50_ndcir': {'sheet': '50% SOC nDCIR', 'y_col': 'nDCIR', 'y_name': 'SOC 50% nDCIR', 'range_key': 'soc50_ndcir', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
        'soc50_dcir0': {'sheet': '50% SOC DCIR0', 'y_col': 'DCIR0', 'y_name': 'SOC 50% DCIR0', 'range_key': 'soc50_dcir0', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
        'soc50_dcirt': {'sheet': '50% SOC DCIRt', 'y_col': 'DCIRt', 'y_name': 'SOC 50% DCIRt', 'range_key': 'soc50_dcirt', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
        'soc50_ndcir0': {'sheet': '50% SOC nDCIR0', 'y_col': 'nDCIR0', 'y_name': 'SOC 50% nDCIR0', 'range_key': 'soc50_ndcir0', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
        'soc50_ndcirt': {'sheet': '50% SOC nDCIRt', 'y_col': 'nDCIRt', 'y_name': 'SOC 50% nDCIRt', 'range_key': 'soc50_ndcirt', 'df_key': 'soc50', 'xmax_key': 'soc50_cycle'},
    }

    worksheets_1c = {}
    charts_1c = {}
    chart_has_series_1c = {}

    for spec_key, spec in plot_1c_specs.items():
        worksheets_1c[spec_key] = workbook.add_chartsheet(make_safe_chart_sheet_name(spec['title']))
        charts_1c[spec_key] = workbook.add_chart({'type': 'scatter', 'subtype': 'straight'})
        chart_has_series_1c[spec_key] = False

    for spec in main_specs.values():
        ensure_chart(spec['sheet'])
    for spec in soc_specs.values():
        ensure_chart(spec['sheet'])

    metric_data = {k: {} for k in metric_map.keys()}
    c10_retention_data = {}
    summary_feature_rows = []
    summary_c10_rows = []
    cell_table_store = {}
    file_order_cells = []

    lower_rce_threshold = 98
    upper_rce_threshold = 102
    lower_hi1_threshold = np.log(np.exp(98 / 100) - 1)
    upper_hi1_threshold = np.log(np.exp(102 / 100) - 1)

    for file_idx in tqdm(range(len(files))):
        file = files[file_idx]
        cell = file.split('.')[0]
        file_order_cells.append(cell)

        raw_data = NewareNDA.read(raw.current_directory + '/' + file)
        statis_data = get_statis(raw_data)

        statis_sub = statis_data[statis_data['Status'].isin(['CCCV_Chg', 'CC_Chg'])]
        step_times = statis_sub.groupby(['Cycle'])['Step time'].sum()
        acr_chg_cycle = statis_sub['Cycle']
        acr = 1 / (step_times / 3600)
        acr = acr[~acr.index.isin(soc)].values.astype(np.float64)

        total_cap = statis_sub.groupby('Cycle')[['Charge Capacity(mAh)', 'Step time']].sum().reset_index()
        total_cap.columns = ['Cycle', 'Total Charge Capacity(mAh)', 'Total Step time']
        total_cap['Cycle'] = pd.to_numeric(total_cap['Cycle'], errors='coerce')
        total_cap['Total Charge Capacity(mAh)'] = pd.to_numeric(total_cap['Total Charge Capacity(mAh)'], errors='coerce').astype('float64')
        total_cap['Total Step time'] = pd.to_numeric(total_cap['Total Step time'], errors='coerce').astype('float64')
        total_cap[f'{acr_string}% ACR'] = total_cap['Total Charge Capacity(mAh)'] * acr_percent

        acr_list = []
        acr_subset_cycle = np.intersect1d(raw_data['Cycle'].unique(), acr_chg_cycle)
        for cycle in acr_subset_cycle:
            temp_cycle_df = raw_data[(raw_data.Cycle == cycle) & (raw_data.Status.isin(['CC_Chg', 'CCCV_Chg']))].copy()
            temp_cycle_df = adjust_time(temp_cycle_df, 'Time', 'Charge time(s)')
            temp_cycle_df = adjust_time(temp_cycle_df, 'Charge_Capacity(mAh)', 'Cumulative Charge Capacity(mAh)')
            small_capacity = total_cap[total_cap.Cycle == cycle][f'{acr_string}% ACR'].values[0]
            f = interp1d(
                temp_cycle_df['Cumulative Charge Capacity(mAh)'],
                temp_cycle_df['Charge time(s)'],
                bounds_error=False
            )
            acr_list.append(float(f(small_capacity)))

        total_cap[f'{acr_string}% Time'] = pd.Series(acr_list, dtype='float64')
        total_cap['20% Time'] = total_cap['Total Step time'] - total_cap[f'{acr_string}% Time']
        total_cap['20% Time Ratio'] = safe_divide(
            total_cap['20% Time'],
            total_cap['Total Step time'],
            fill_value=0.0
        )
        total_cap[f'{acr_string}% ACR'] = acr_percent / (np.array(acr_list, dtype=np.float64) / 3600)

        filtered = statis_data[~statis_data['Cycle'].isin(soc)].reset_index(drop=True).copy()

        if 'CC_DChg' not in filtered[filtered['Cycle'] == filtered['Cycle'].max()]['Status'].values[:]:
            last_row = filtered.iloc[-1].copy()
            new_row = pd.Series(index=filtered.columns, dtype='object')

            for col in filtered.columns:
                new_row[col] = np.nan

            new_row.iloc[0] = last_row.iloc[0] + 1
            new_row.iloc[1] = last_row.iloc[1]
            new_row.iloc[6] = 'CC_DChg'
            new_row.iloc[12] = last_row.iloc[12]
            new_row.iloc[13] = last_row.iloc[13]

            filtered = pd.concat([filtered, new_row.to_frame().T], ignore_index=True)

        corrected_cycles = [i for i in range(1, len(filtered['Cycle'].unique()) + 1)]
        neware_cycles = filtered['Cycle'].unique()
        corrected_map = {nw: corr for nw, corr in zip(neware_cycles, corrected_cycles)}
        filtered["Corrected Cycle"] = filtered["Cycle"].map(corrected_map)

        Charge_capacity = filtered[filtered['Status'].isin(['CCCV_Chg', 'CC_Chg'])].groupby('Corrected Cycle')['Charge Capacity(mAh)'].sum()
        Discharge_capacity = filtered[filtered['Status'].isin(['CCCV_DChg', 'CC_DChg'])].groupby('Corrected Cycle')['Discharge Capacity(mAh)'].sum()
        Discharge_energy = filtered[filtered['Status'].isin(['CCCV_DChg', 'CC_DChg'])].groupby('Corrected Cycle')['Discharge Energy(mWh)'].sum()
        Charge_energy = filtered[filtered['Status'].isin(['CCCV_Chg', 'CC_Chg'])].groupby('Corrected Cycle')['Charge Energy(mWh)'].sum()
        Charge_resistances = filtered[filtered['Status'].isin(['CCCV_Chg', 'CC_Chg'])].groupby('Corrected Cycle')['DCIR(Ohm)'].sum()
        Discharge_resistances = filtered[filtered['Status'].isin(['CCCV_DChg', 'CC_DChg'])].groupby('Corrected Cycle')['DCIR(Ohm)'].sum()

        rest_charged_steps, previous_row = [], ''
        for _, row in filtered.iterrows():
            if row['Status'] == 'Rest' and previous_row == 'CCCV_Chg':
                rest_charged_steps.append(row['Step'])
            previous_row = row['Status']
        rest_steps = filtered[filtered.Step.isin(rest_charged_steps)]
        dV_charged = np.zeros(len(corrected_cycles), dtype=np.float64)
        dV_charged[:len(rest_steps)] = (rest_steps['End Voltage(V)'] - rest_steps['Start Voltage(V)']).to_numpy(dtype=np.float64)
        normal_dV_charged = dV_charged / dV_charged[0] if len(dV_charged) > 0 and dV_charged[0] != 0 else np.full(len(dV_charged), np.nan, dtype=np.float64)

        rest_discharged_steps, dis_previous_row = [], ''
        for _, row in filtered.iterrows():
            if row['Status'] == 'Rest' and dis_previous_row == 'CC_DChg':
                rest_discharged_steps.append(row['Step'])
            dis_previous_row = row['Status']
        dis_rest_steps = filtered[filtered.Step.isin(rest_discharged_steps)]
        dV_discharged = np.zeros(len(corrected_cycles) + 10, dtype=np.float64)
        dV_discharged[:len(dis_rest_steps)] = (dis_rest_steps['End Voltage(V)'] - dis_rest_steps['Start Voltage(V)']).to_numpy(dtype=np.float64)

        min_len = min(len(dV_discharged), len(corrected_cycles))
        dV_discharged = dV_discharged[:min_len]
        normal_dV_discharged = dV_discharged / dV_discharged[0] if len(dV_discharged) > 0 and dV_discharged[0] != 0 else np.full(len(dV_discharged), np.nan, dtype=np.float64)
        corrected_cycles = corrected_cycles[:min_len]
        neware_cycles = neware_cycles[:min_len]

        df = pd.DataFrame({
            'Corrected Cycle': pd.Series(corrected_cycles, dtype='float64'),
            'Cycle': pd.Series(neware_cycles, dtype='float64'),
            'Charge Capacity': pd.Series(Charge_capacity.values[:min_len], dtype='float64'),
            'Discharge Capacity': pd.Series(Discharge_capacity.values[:min_len], dtype='float64'),
        })

        charge_energy_arr = pd.Series(Charge_energy.values[:min_len], dtype='float64')
        discharge_energy_arr = pd.Series(Discharge_energy.values[:min_len], dtype='float64')
        charge_res_arr = pd.Series(Charge_resistances.values[:min_len], dtype='float64')
        discharge_res_arr = pd.Series(Discharge_resistances.values[:min_len], dtype='float64')

        df['Charge Energy (mWh)'] = charge_energy_arr
        df['Discharge Energy (mWh)'] = discharge_energy_arr
        df['Charge Resistance'] = charge_res_arr
        df['Discharge Resistance'] = discharge_res_arr
        df['100% ACR'] = pd.Series(acr[:min_len], dtype='float64')
        df['Delta V Charge'] = pd.Series(dV_charged, dtype='float64')
        df['Delta V Discharge'] = pd.Series(dV_discharged, dtype='float64')
        df['nDelta V Charge'] = pd.Series(normal_dV_charged, dtype='float64')
        df['nDelta V Discharge'] = pd.Series(normal_dV_discharged, dtype='float64')

        df['Average Charge Voltage'] = safe_divide(df['Charge Energy (mWh)'], df['Charge Capacity'])
        df['Average Discharge Voltage'] = safe_divide(df['Discharge Energy (mWh)'], df['Discharge Capacity'])

        first_discharge = df['Discharge Capacity'].iloc[0] if len(df) > 0 else np.nan
        first_charge_energy = df['Charge Energy (mWh)'].iloc[0] if len(df) > 0 else np.nan
        first_discharge_energy = df['Discharge Energy (mWh)'].iloc[0] if len(df) > 0 else np.nan
        first_charge_res = df['Charge Resistance'].iloc[0] if len(df) > 0 else np.nan
        first_discharge_res = df['Discharge Resistance'].iloc[0] if len(df) > 0 else np.nan

        df['Cycle Life'] = np.nan if pd.isna(first_discharge) or first_discharge == 0 else (df['Discharge Capacity'] / first_discharge) * 100
        df['Coulombic Efficiency (%)'] = safe_divide(df['Discharge Capacity'], df['Charge Capacity']) * 100

        prev_discharge = df['Discharge Capacity'].shift(1)
        df['RCE'] = safe_divide(df['Charge Capacity'], prev_discharge) * 100
        df['RCE'] = as_float64_series(df['RCE'])
        if len(df) > 0:
            df.at[0, 'RCE'] = np.float64(100.0)

        rce_for_hi1 = as_float64_series(df['RCE']) / np.float64(100.0)
        df['HI1'] = pd.Series(
            np.log(np.exp(rce_for_hi1) - 1),
            index=df.index,
            dtype='float64'
        )
        if len(df) > 0:
            df.at[0, 'HI1'] = np.float64(np.log(np.exp(1.0) - 1.0))

        df = df.replace([np.inf, -np.inf], np.nan)

        df['nCharge Resistance'] = np.nan if pd.isna(first_charge_res) or first_charge_res == 0 else df['Charge Resistance'] / first_charge_res
        df['nDischarge Resistance'] = np.nan if pd.isna(first_discharge_res) or first_discharge_res == 0 else df['Discharge Resistance'] / first_discharge_res
        df['Charge Energy Retention'] = np.nan if pd.isna(first_charge_energy) or first_charge_energy == 0 else (df['Charge Energy (mWh)'] / first_charge_energy) * 100
        df['Discharge Energy Retention'] = np.nan if pd.isna(first_discharge_energy) or first_discharge_energy == 0 else (df['Discharge Energy (mWh)'] / first_discharge_energy) * 100
        df['Energy Efficiency %'] = safe_divide(df['Discharge Energy (mWh)'], df['Charge Energy (mWh)']) * 100
        df['Voltage Slippage'] = 0.5 * (df['Average Charge Voltage'] + df['Average Discharge Voltage'])
        df['V Polarization'] = df['Average Charge Voltage'] - df['Average Discharge Voltage']

        group_name_series = metadata.loc[metadata['Barcode'] == cell, 'Group']
        group_name = group_name_series.iloc[0] if not group_name_series.empty else 'Unknown'
        color_final = metadata[metadata['Barcode'] == cell]['Color'].values[0]

        df = attach_cycle_map_info(df, cycle_map_df)

        for c in df.columns:
            if c not in ['Cycle Note', 'Cycle State']:
                df[c] = pd.to_numeric(df[c], errors='coerce')

        numeric_cols_df = [c for c in df.columns if c not in ['Cycle Note', 'Cycle State']]
        for c in numeric_cols_df:
            df[c] = df[c].astype('float64')

        # 1C full / excl first
        df_1c_all = df.dropna(subset=['1C Cycle']).copy()

        df_1c_ex1 = df.dropna(subset=['1C Cycle Excl First']).copy()
        if len(df_1c_ex1) > 0:
            first_1c_discharge = df_1c_ex1['Discharge Capacity'].iloc[0]
            first_1c_charge_energy = df_1c_ex1['Charge Energy (mWh)'].iloc[0]
            first_1c_discharge_energy = df_1c_ex1['Discharge Energy (mWh)'].iloc[0]

            df_1c_ex1['Cycle Life 1C'] = 100 * df_1c_ex1['Discharge Capacity'] / first_1c_discharge if pd.notna(first_1c_discharge) and first_1c_discharge != 0 else np.nan
            df_1c_ex1['Charge Energy Retention 1C'] = 100 * df_1c_ex1['Charge Energy (mWh)'] / first_1c_charge_energy if pd.notna(first_1c_charge_energy) and first_1c_charge_energy != 0 else np.nan
            df_1c_ex1['Discharge Energy Retention 1C'] = 100 * df_1c_ex1['Discharge Energy (mWh)'] / first_1c_discharge_energy if pd.notna(first_1c_discharge_energy) and first_1c_discharge_energy != 0 else np.nan

            df_1c_ex1['CE Loss'] = 100 - df_1c_ex1['Coulombic Efficiency (%)']
            df_1c_ex1['Cumulative CE Loss'] = df_1c_ex1['CE Loss'].cumsum()

            df_1c_ex1['RCE Loss'] = df_1c_ex1['RCE'] - 100
            df_1c_ex1['Cumulative RCE Loss'] = df_1c_ex1['RCE Loss'].cumsum()

            numeric_cols_1c = [c for c in df_1c_ex1.columns if c not in ['Cycle Note', 'Cycle State']]
            for c in numeric_cols_1c:
                df_1c_ex1[c] = pd.to_numeric(df_1c_ex1[c], errors='coerce').astype('float64')

        # C/10
        note_series = df['Cycle Note'].astype(str).str.strip().str.upper()
        df_c10 = df[note_series.str.contains(r'C\s*/\s*10|0\.1C', regex=True, na=False)].copy()

        if len(df_c10) > 0:
            df_c10 = df_c10.sort_values('Map Corrected Cycle').reset_index(drop=True)
            df_c10['C/10 Cycle #'] = np.arange(1, len(df_c10) + 1, dtype=np.float64)
            first_c10_discharge = df_c10['Discharge Capacity'].iloc[0]
            df_c10['C/10 Capacity Retention (%)'] = 100 * df_c10['Discharge Capacity'] / first_c10_discharge if pd.notna(first_c10_discharge) and first_c10_discharge != 0 else np.nan
            update_range(ranges, 'c10_retention', df_c10['C/10 Capacity Retention (%)'])
            update_max(max_tracker, 'c10_cycle', df_c10['Map Corrected Cycle'].max())
            c10_retention_data[(group_name, cell)] = df_c10.set_index('Map Corrected Cycle')['C/10 Capacity Retention (%)'].copy()

        # kinetic
        df_kinetic = pd.DataFrame()
        if len(df_c10) >= 2 and len(df_1c_all) > 0:
            c10_points = df_c10[['Map Corrected Cycle', 'Discharge Capacity']].copy()
            c10_points = c10_points.sort_values('Map Corrected Cycle').reset_index(drop=True)

            one_c_points = df_1c_all[['Map Corrected Cycle', 'Discharge Capacity', '1C Cycle']].copy()
            one_c_points = one_c_points.sort_values('Map Corrected Cycle').reset_index(drop=True)

            kinetic_rows = []

            for i in range(1, len(c10_points)):
                start_c10_cycle = c10_points.loc[i - 1, 'Map Corrected Cycle']
                end_c10_cycle = c10_points.loc[i, 'Map Corrected Cycle']

                start_c10_cap = c10_points.loc[i - 1, 'Discharge Capacity']
                end_c10_cap = c10_points.loc[i, 'Discharge Capacity']

                if pd.notna(start_c10_cap) and start_c10_cap != 0 and pd.notna(end_c10_cap):
                    c10_ret = 100 * end_c10_cap / start_c10_cap
                else:
                    c10_ret = np.nan

                one_c_window = one_c_points[
                    (one_c_points['Map Corrected Cycle'] > start_c10_cycle) &
                    (one_c_points['Map Corrected Cycle'] < end_c10_cycle)
                ].copy()

                if len(one_c_window) > 0:
                    one_c_window = one_c_window.sort_values('Map Corrected Cycle').reset_index(drop=True)

                    start_1c_cycle = one_c_window.loc[0, 'Map Corrected Cycle']
                    end_1c_cycle = one_c_window.loc[len(one_c_window) - 1, 'Map Corrected Cycle']
                    start_1c_cap = one_c_window.loc[0, 'Discharge Capacity']
                    end_1c_cap = one_c_window.loc[len(one_c_window) - 1, 'Discharge Capacity']
                    start_1c_cycle_number = one_c_window.loc[0, '1C Cycle']
                    end_1c_cycle_number = one_c_window.loc[len(one_c_window) - 1, '1C Cycle']

                    if pd.notna(start_1c_cap) and start_1c_cap != 0 and pd.notna(end_1c_cap):
                        one_c_ret = 100 * end_1c_cap / start_1c_cap
                    else:
                        one_c_ret = np.nan
                else:
                    start_1c_cycle = np.nan
                    end_1c_cycle = np.nan
                    start_1c_cycle_number = np.nan
                    end_1c_cycle_number = np.nan
                    one_c_ret = np.nan

                kinetic_fade = c10_ret - one_c_ret if pd.notna(one_c_ret) and pd.notna(c10_ret) else np.nan
                kinetic_rows.append({
                    'Interval #': i,
                    'Start C/10 Map Cycle': start_c10_cycle,
                    'End C/10 Map Cycle': end_c10_cycle,
                    'Start 1C Map Cycle': start_1c_cycle,
                    'End 1C Map Cycle': end_1c_cycle,
                    'Start 1C Cycle #': start_1c_cycle_number,
                    'End 1C Cycle #': end_1c_cycle_number,
                    'C/10 Retention (%)': c10_ret,
                    '1C Retention (%)': one_c_ret,
                    'Kinetic Fading (%)': kinetic_fade,
                })

            df_kinetic = pd.DataFrame(kinetic_rows)
            if len(df_kinetic) > 0:
                for c in df_kinetic.columns:
                    df_kinetic[c] = pd.to_numeric(df_kinetic[c], errors='coerce').astype('float64')
                update_range(ranges, 'kinetic_fade', df_kinetic['Kinetic Fading (%)'])
                update_max(max_tracker, 'kinetic_cycle', df_kinetic['End C/10 Map Cycle'].max())

        # total_cap 1C excl first
        cm_map = cycle_map_df.drop_duplicates(subset=['NW Cycle']).copy()
        total_cap_1c_ex1 = total_cap.copy()
        cycle_map_dict_ex1 = dict(zip(cm_map['NW Cycle'], cm_map['1C Cycle Excl First']))
        total_cap_1c_ex1['1C Cycle Excl First'] = pd.to_numeric(total_cap_1c_ex1['Cycle'], errors='coerce').map(cycle_map_dict_ex1)
        total_cap_1c_ex1 = total_cap_1c_ex1.dropna(subset=['1C Cycle Excl First']).copy()

        numeric_cols_total_cap_1c = [c for c in total_cap_1c_ex1.columns]
        for c in numeric_cols_total_cap_1c:
            total_cap_1c_ex1[c] = pd.to_numeric(total_cap_1c_ex1[c], errors='coerce')

        # summary row
        electrolyte_series = metadata.loc[metadata['Barcode'] == cell, 'Group']
        electrolyte_name = electrolyte_series.iloc[0] if not electrolyte_series.empty else 'Unknown'

        final_1c_cycle = get_final_1c_cycle_number(df_1c_all, cycle_col='1C Cycle')
        final_1c_cycle_minus_50 = final_1c_cycle - 50 if pd.notna(final_1c_cycle) and final_1c_cycle > 100 else np.nan

        summary_row = {
            'Barcode': cell,
            'Electrolyte': electrolyte_name,
            'Final 1C Cycle Number': final_1c_cycle,
            'Final 1C Cycle Number - 50': final_1c_cycle_minus_50,
        }

        for metric_key, metric_info in summary_1c_metrics.items():
            source_name = metric_info['source']
            col_name = metric_info['col']

            if source_name == 'df_1c_ex1':
                source_df = df_1c_ex1
            elif source_name == 'total_cap_1c_ex1':
                source_df = total_cap_1c_ex1
            else:
                source_df = pd.DataFrame()

            feats = extract_1c_features_extended_from_df(source_df, col_name, cycle_col='1C Cycle Excl First')

            summary_row[f'{metric_key}_f_1cyc'] = feats['f_1cyc']
            summary_row[f'{metric_key}_f_50cyc'] = feats['f_50cyc']
            summary_row[f'{metric_key}_f_endm50cyc'] = feats['f_endm50cyc']
            summary_row[f'{metric_key}_f_endm1cyc'] = feats['f_endm1cyc']
            summary_row[f'{metric_key}_df_dcyc_1_50'] = feats['df_dcyc_1_50']
            summary_row[f'{metric_key}_df_dcyc_50_endm50'] = feats['df_dcyc_50_endm50']
            summary_row[f'{metric_key}_df_dcyc_endm50_endm1'] = feats['df_dcyc_endm50_endm1']
            summary_row[f'{metric_key}_df_dcyc_1_endm1'] = feats['df_dcyc_1_endm1']
            summary_row[f'{metric_key}_abruptness'] = feats['abruptness']

        for metric_key, metric_info in summary_ce_like_metrics.items():
            source_name = metric_info['source']
            col_name = metric_info['col']

            if source_name == 'df_1c_ex1':
                source_df = df_1c_ex1
            elif source_name == 'total_cap_1c_ex1':
                source_df = total_cap_1c_ex1
            else:
                source_df = pd.DataFrame()

            feats = extract_ce_like_features(source_df, col_name, cycle_col='1C Cycle Excl First')

            summary_row[f'{metric_key}_f_1cyc'] = feats['f_1cyc']
            summary_row[f'{metric_key}_avg_1_25'] = feats['avg_1_25']
            summary_row[f'{metric_key}_avg_25_100'] = feats['avg_25_100']
            summary_row[f'{metric_key}_avg_100_endm50'] = feats['avg_100_endm50']
            summary_row[f'{metric_key}_avg_endm50_endm1'] = feats['avg_endm50_endm1']

        summary_feature_rows.append(summary_row)

        # metric wide tables
        df_idx = df.set_index('Corrected Cycle')
        for display_name, col_name in metric_map.items():
            if col_name in df_idx.columns:
                metric_data[display_name][(group_name, cell)] = df_idx[col_name].copy()

        if len(df_kinetic) > 0 and 'Kinetic Fading (%)' in df_kinetic.columns:
            kinetic_idx = df_kinetic.set_index('End C/10 Map Cycle')
            metric_data['Kinetic Fading (%)'][(group_name, cell)] = kinetic_idx['Kinetic Fading (%)'].copy()

        # -------------------------
        # DCIR (SOC50 only)
        # -------------------------
        dcir = statis_data[statis_data['Cycle'].isin(all_soc)].reset_index(drop=True)
        dcir["Corrected Cycle"] = dcir["Cycle"].map(socs)
        dcir["SOC State"] = dcir["Cycle"].map(socs_states)

        dcir_discharge_all = dcir[dcir['Status'].isin(['CC_DChg', 'CCCV_DChg'])]
        dcir_restbeforedcir = dcir[dcir['Status'] == 'Rest']
        dcir_discharge_all_30sec = dcir_discharge_all[dcir_discharge_all["Step time"] == 30.0]
        dcir_restbeforedcir_1200s = dcir_restbeforedcir[dcir_restbeforedcir["Step time"] == 1200.0]

        groups = dcir_discharge_all_30sec.filter(items=['End Current(mA)', 'End Voltage(V)', 'Corrected Cycle', 'SOC State']).reset_index(drop=True)
        groups1 = dcir_restbeforedcir_1200s.filter(items=['End Current(mA)', 'End Voltage(V)', 'Corrected Cycle', 'SOC State']).reset_index(drop=True)
        groups_DCIR0_discharge = dcir_discharge_all_30sec.filter(items=['Start Current(mA)', 'Start Voltage(V)', 'End Current(mA)', 'End Voltage(V)', 'Corrected Cycle', 'SOC State']).reset_index(drop=True)
        groups_DCIR0_rest = dcir_restbeforedcir_1200s.filter(items=['End Current(mA)', 'End Voltage(V)', 'Corrected Cycle', 'SOC State']).reset_index(drop=True)

        groups['End Voltage(V)'] = pd.to_numeric(groups['End Voltage(V)'], errors='coerce') - pd.to_numeric(groups1['End Voltage(V)'], errors='coerce')
        groups['DCIR'] = 1000 * safe_divide(groups['End Voltage(V)'], groups['End Current(mA)'])
        groups['DCIR0'] = 1000 * safe_divide(
            pd.to_numeric(groups_DCIR0_discharge['Start Voltage(V)'], errors='coerce') - pd.to_numeric(groups_DCIR0_rest['End Voltage(V)'], errors='coerce'),
            groups_DCIR0_discharge['Start Current(mA)']
        )
        groups['DCIRt'] = groups['DCIR'] - groups['DCIR0']

        dcir_frames = {
            'soc50': groups[groups['SOC State'] == 'SOC 50%'].reset_index(drop=True),
        }

        if len(dcir_frames['soc50']) > 0:
            dcir_frames['soc50'] = dcir_frames['soc50'].drop(columns=['End Current(mA)', 'End Voltage(V)'])

        built = {
            'soc50': build_dcir_subset(dcir_frames['soc50'], include_dcir0=True),
        }

        if built['soc50'] is not None:
            exists['soc50'] = True
            update_max(max_tracker, 'soc50_cycle', built['soc50']['Corrected Cycle'].max())

            for rk, col in [
                ('soc50_dcir', 'DCIR'),
                ('soc50_ndcir', 'nDCIR'),
                ('soc50_dcir0', 'DCIR0'),
                ('soc50_dcirt', 'DCIRt'),
                ('soc50_ndcir0', 'nDCIR0'),
                ('soc50_ndcirt', 'nDCIRt')
            ]:
                update_range(ranges, rk, built['soc50'][col])

        summary_c10_row = build_c10_summary_row(
            cell=cell,
            electrolyte_name=electrolyte_name,
            df_c10=df_c10,
            df_kinetic=df_kinetic,
            soc50_df=built['soc50'],
        )
        summary_c10_rows.append(summary_c10_row)

        # global ranges
        for rk, col in [
            ('charge', 'Charge Capacity'),
            ('discharge', 'Discharge Capacity'),
            ('charge_energy', 'Charge Energy (mWh)'),
            ('discharge_energy', 'Discharge Energy (mWh)'),
            ('charge_energy_retention', 'Charge Energy Retention'),
            ('discharge_energy_retention', 'Discharge Energy Retention'),
            ('avg_charge', 'Average Charge Voltage'),
            ('avg_discharge', 'Average Discharge Voltage'),
            ('ce', 'Coulombic Efficiency (%)'),
            ('res_chg', 'Charge Resistance'),
            ('res_dchg', 'Discharge Resistance'),
            ('nres_chg', 'nCharge Resistance'),
            ('nres_dchg', 'nDischarge Resistance'),
            ('acr', '100% ACR'),
            ('dv_chg', 'Delta V Charge'),
            ('ndv_chg', 'nDelta V Charge'),
            ('dv_dchg', 'Delta V Discharge'),
            ('ndv_dchg', 'nDelta V Discharge'),
            ('energy_eff', 'Energy Efficiency %'),
            ('vslip', 'Voltage Slippage'),
            ('vpol', 'V Polarization'),
        ]:
            update_range(ranges, rk, df[col])

        update_range(ranges, 'custom_acr', total_cap[f'{acr_string}% ACR'])
        update_range(ranges, 'ratio20', total_cap['20% Time Ratio'])

        if len(df_1c_ex1) > 0:
            update_range(ranges, 'cum_ce_loss', df_1c_ex1['Cumulative CE Loss'])
            update_range(ranges, 'cum_rce_loss', df_1c_ex1['Cumulative RCE Loss'])

        update_max(max_tracker, 'cycle', df['Corrected Cycle'].max())
        update_max(max_tracker, 'cycle_life', df['Cycle Life'].max())
        update_max(max_tracker, 'cycle_1c_ex1', df['1C Cycle Excl First'].max(skipna=True))
        update_max(max_tracker, 'cycle_1c_ex1', total_cap_1c_ex1['1C Cycle Excl First'].max(skipna=True))

        temp_rce = clean_series(df['RCE'])
        if len(temp_rce) > 0:
            temp_min_rce, temp_max_rce = temp_rce.min(), temp_rce.max()
            ranges['rce']['min'] = temp_min_rce if temp_min_rce < lower_rce_threshold else lower_rce_threshold
            ranges['rce']['max'] = temp_max_rce if temp_max_rce > upper_rce_threshold else upper_rce_threshold

        temp_hi1 = clean_series(df['HI1'])
        if len(temp_hi1) > 0:
            temp_min_hi1, temp_max_hi1 = temp_hi1.min(), temp_hi1.max()
            ranges['hi1']['min'] = temp_min_hi1 if temp_min_hi1 < lower_hi1_threshold else lower_hi1_threshold
            ranges['hi1']['max'] = temp_max_hi1 if temp_max_hi1 > upper_hi1_threshold else upper_hi1_threshold

        # -------------------------
        # Build consolidated tables
        # -------------------------
        sheet_full = make_safe_sheet_name(cell)
        sheet_1c_ex1 = make_safe_sheet_name(f'{cell}_1C_ExclFirst')
        sheet_c10 = make_safe_sheet_name(f'{cell}_C10')

        df_full_tab = build_full_cell_table(df, total_cap, acr_string)
        df_1c_ex1_tab = build_1c_ex1_table(df_1c_ex1, total_cap_1c_ex1, acr_string)
        df_c10_tab = build_c10_combined_table(df_c10, df_kinetic, built['soc50'])

        cell_table_store[cell] = {
            'full_sheet': sheet_full,
            'full_df': df_full_tab,
            'ex1_sheet': sheet_1c_ex1,
            'ex1_df': df_1c_ex1_tab,
            'c10_sheet': sheet_c10,
            'c10_df': df_c10_tab,
        }

        # -------------------------
        # Main charts
        # -------------------------
        for k, spec in main_specs.items():
            if spec.get('source') == 'total_cap':
                if len(df_full_tab) > 0 and spec['y_col'] in df_full_tab.columns:
                    add_df_series(
                        charts[spec['sheet']],
                        df_full_tab,
                        sheet_full,
                        'Cycle',
                        spec['y_col'],
                        color_final,
                        group_name
                    )

            elif spec.get('source') == 'c10':
                c10_plot_col = f'C10 | {spec["y_col"]}'
                c10_x_col = 'C10 | Map Corrected Cycle'
                if len(df_c10_tab) > 0 and c10_plot_col in df_c10_tab.columns and c10_x_col in df_c10_tab.columns:
                    add_df_series(
                        charts[spec['sheet']],
                        df_c10_tab,
                        sheet_c10,
                        c10_x_col,
                        c10_plot_col,
                        color_final,
                        group_name
                    )
                    chart_has_series_c10 = True

            elif spec.get('source') == 'kinetic':
                kinetic_plot_col = f'Kinetic | {spec["y_col"]}'
                kinetic_x_col = 'Kinetic | End C/10 Map Cycle'
                if len(df_c10_tab) > 0 and kinetic_plot_col in df_c10_tab.columns and kinetic_x_col in df_c10_tab.columns:
                    add_df_series(
                        charts[spec['sheet']],
                        df_c10_tab,
                        sheet_c10,
                        kinetic_x_col,
                        kinetic_plot_col,
                        color_final,
                        group_name
                    )
                    chart_has_series_kinetic = True

            else:
                if len(df_full_tab) > 0 and spec['y_col'] in df_full_tab.columns:
                    add_df_series(
                        charts[spec['sheet']],
                        df_full_tab,
                        sheet_full,
                        'Corrected Cycle',
                        spec['y_col'],
                        color_final,
                        group_name
                    )

        # -------------------------
        # SOC50 charts
        # -------------------------
        for k, spec in soc_specs.items():
            prefixed_col = f'SOC50 | {spec["y_col"]}'
            x_col = 'SOC50 | DCIR Cycle'

            if len(df_c10_tab) > 0 and prefixed_col in df_c10_tab.columns and x_col in df_c10_tab.columns:
                add_df_series(
                    charts[spec['sheet']],
                    df_c10_tab,
                    sheet_c10,
                    x_col,
                    prefixed_col,
                    color_final,
                    group_name
                )

        # -------------------------
        # 1C excl first charts
        # -------------------------
        for spec_key, spec in plot_1c_specs.items():
            if len(df_1c_ex1_tab) == 0:
                continue
            if spec['df_col'] not in df_1c_ex1_tab.columns:
                continue

            add_df_series(
                charts_1c[spec_key],
                df_1c_ex1_tab,
                sheet_1c_ex1,
                '1C Cycle Excl First',
                spec['df_col'],
                color_final,
                group_name
            )
            chart_has_series_1c[spec_key] = True

    # -------------------------
    # thresholds sheet
    # -------------------------
    cycle_count = int(max_tracker['cycle']) if pd.notna(max_tracker['cycle']) else 0
    
    thresholds = pd.DataFrame({
        'Cycle': np.arange(1, cycle_count + 1, dtype=np.int64),
        'Lower RCE': np.full(cycle_count, lower_rce_threshold, dtype=np.float64),
        'Upper RCE': np.full(cycle_count, upper_rce_threshold, dtype=np.float64),
        'Lower HI1': np.full(cycle_count, lower_hi1_threshold, dtype=np.float64),
        'Upper HI1': np.full(cycle_count, upper_hi1_threshold, dtype=np.float64)
    })
    if 'RCE' in charts:
        charts['RCE'].add_series({
            'categories': ['thresholds', 1, 0, len(thresholds), 0],
            'values': ['thresholds', 1, 1, len(thresholds), 1],
            'line': {'color': 'red', 'dash_type': 'dash'},
            'name': 'lower bound',
        })
        charts['RCE'].add_series({
            'categories': ['thresholds', 1, 0, len(thresholds), 0],
            'values': ['thresholds', 1, 2, len(thresholds), 2],
            'line': {'color': 'red', 'dash_type': 'dash'},
            'name': 'upper bound',
        })

    if 'HI1' in charts:
        charts['HI1'].add_series({
            'categories': ['thresholds', 1, 0, len(thresholds), 0],
            'values': ['thresholds', 1, 3, len(thresholds), 3],
            'line': {'color': 'red', 'dash_type': 'dash'},
            'name': 'lower bound',
        })
        charts['HI1'].add_series({
            'categories': ['thresholds', 1, 0, len(thresholds), 0],
            'values': ['thresholds', 1, 4, len(thresholds), 4],
            'line': {'color': 'red', 'dash_type': 'dash'},
            'name': 'upper bound',
        })

    # -------------------------
    # axis setting
    # -------------------------
    for k, spec in main_specs.items():
        chart_obj = charts.get(spec['sheet'], None)
        if chart_obj is None:
            continue

        if k == 'cycle_life':
            y_min, y_max = 70, max_tracker['cycle_life']
            x_max = max_tracker['cycle']
            set_chart_axis(chart_obj, 'Cycle Number', spec['y_name'], 1, x_max, y_min, y_max, remove_legends)
            continue

        if k == 'c10_retention':
            y_min = ranges['c10_retention']['min'] if ranges['c10_retention']['min'] != 100000 else 0
            y_max = ranges['c10_retention']['max'] if ranges['c10_retention']['max'] > 0 else 100
            x_max = max_tracker['c10_cycle'] if max_tracker['c10_cycle'] > 0 else 1
            set_chart_axis(chart_obj, 'Corrected Cycle', spec['y_name'], 1, x_max, y_min, y_max, remove_legends)
            continue

        if k == 'kinetic_fade':
            y_min = ranges['kinetic_fade']['min'] if ranges['kinetic_fade']['min'] != 100000 else 0
            y_max = ranges['kinetic_fade']['max'] if ranges['kinetic_fade']['max'] > 0 else 1
            x_max = max_tracker['kinetic_cycle'] if max_tracker['kinetic_cycle'] > 0 else 1
            set_chart_axis(chart_obj, 'Map Corrected Cycle', spec['y_name'], 1, x_max, y_min, y_max, remove_legends)
            continue

        y_min, y_max = ranges[spec['range_key']]['min'], ranges[spec['range_key']]['max']
        x_max = max_tracker['cycle']
        set_chart_axis(chart_obj, 'Cycle Number', spec['y_name'], 1, x_max, y_min, y_max, remove_legends)

    for k, spec in soc_specs.items():
        chart_obj = charts.get(spec['sheet'], None)
        if chart_obj is None:
            continue
        if not exists[spec['df_key']]:
            continue
        set_chart_axis(
            chart_obj,
            'Corrected Cycle',
            spec['y_name'],
            1,
            max_tracker[spec['xmax_key']] if max_tracker[spec['xmax_key']] > 0 else 1,
            ranges[spec['range_key']]['min'],
            ranges[spec['range_key']]['max'],
            remove_legends
        )

    for spec_key, spec in plot_1c_specs.items():
        if not chart_has_series_1c.get(spec_key, False):
            continue

        if spec_key == 'cycle_life_ex1':
            y_min, y_max = 70, max_tracker['cycle_life']
        else:
            y_min, y_max = ranges[spec['range_key']]['min'], ranges[spec['range_key']]['max']
            if spec['range_key'] == 'ratio20':
                y_min = y_min if y_min != 1e9 else 0
                y_max = y_max if y_max > 0 else 1

        set_chart_axis(
            charts_1c[spec_key],
            '1C Cycle (Excl First)',
            spec['y_name'],
            1,
            max_tracker['cycle_1c_ex1'] if max_tracker['cycle_1c_ex1'] > 0 else 1,
            y_min,
            y_max,
            remove_legends
        )

    # -------------------------
    # attach charts to chartsheets
    # -------------------------
    for spec_key in plot_1c_specs.keys():
        if chart_has_series_1c.get(spec_key, False):
            worksheets_1c[spec_key].set_chart(charts_1c[spec_key])

    for spec_key, spec in main_specs.items():
        sheet_obj = chartsheets.get(spec['sheet'], None)
        chart_obj = charts.get(spec['sheet'], None)
        if sheet_obj is None or chart_obj is None:
            continue
        if not chart_obj.series:
            continue
        if spec_key == 'c10_retention' and not chart_has_series_c10:
            continue
        if spec_key == 'kinetic_fade' and not chart_has_series_kinetic:
            continue
        sheet_obj.set_chart(chart_obj)

    for spec_key, spec in soc_specs.items():
        sheet_obj = chartsheets.get(spec['sheet'], None)
        chart_obj = charts.get(spec['sheet'], None)
        if sheet_obj is None or chart_obj is None:
            continue
        if not chart_obj.series:
            continue
        if not exists[spec['df_key']]:
            continue
        sheet_obj.set_chart(chart_obj)

    # -------------------------
    # write per-barcode tables after charts
    # -------------------------
    for cell in file_order_cells:
        if cell not in cell_table_store:
            continue
        payload = cell_table_store[cell]
        payload['full_df'].to_excel(writer, sheet_name=payload['full_sheet'], index=False)
        payload['ex1_df'].to_excel(writer, sheet_name=payload['ex1_sheet'], index=False)
        payload['c10_df'].to_excel(writer, sheet_name=payload['c10_sheet'], index=False)

    # -------------------------
    # summary/aggregate sheets
    # -------------------------
    metadata.to_excel(writer, sheet_name='cell_codes', index=False)
    thresholds.to_excel(writer, sheet_name='thresholds', index=False)

    table_sheet_names = []
    for display_name, series_dict in metric_data.items():
        if not series_dict:
            continue
        wide = pd.concat(series_dict, axis=1)
        wide.columns = pd.MultiIndex.from_tuples(wide.columns, names=['Group', 'Barcode'])
        wide.index.name = 'Cycle'
        if display_name == 'Kinetic Fading (%)':
            wide.index.name = '1C Cycle'
        sheet_name = make_safe_sheet_name(f'{display_name} Table')
        wide.to_excel(writer, sheet_name=sheet_name)
        table_sheet_names.append(sheet_name)

    if c10_retention_data:
        wide_c10 = pd.concat(c10_retention_data, axis=1)
        wide_c10.columns = pd.MultiIndex.from_tuples(wide_c10.columns, names=['Group', 'Barcode'])
        wide_c10.index.name = 'Corrected Cycle'
        c10_table_sheet = make_safe_sheet_name('C10 Capacity Retention Table')
        wide_c10.to_excel(writer, sheet_name=c10_table_sheet)
        table_sheet_names.append(c10_table_sheet)

    if summary_feature_rows:
        summary_features_df = pd.DataFrame(summary_feature_rows)

        fixed_cols = ['Barcode', 'Electrolyte', 'Final 1C Cycle Number', 'Final 1C Cycle Number - 50']

        feature_order_general = [
            'f_1cyc',
            'f_50cyc',
            'f_endm50cyc',
            'f_endm1cyc',
            'df_dcyc_1_50',
            'df_dcyc_50_endm50',
            'df_dcyc_endm50_endm1',
            'df_dcyc_1_endm1',
            'abruptness'
        ]

        feature_order_ce_like = [
            'f_1cyc',
            'avg_1_25',
            'avg_25_100',
            'avg_100_endm50',
            'avg_endm50_endm1'
        ]

        ordered_cols = fixed_cols[:]

        for metric_key in summary_1c_metrics.keys():
            for feat in feature_order_general:
                col_name = f'{metric_key}_{feat}'
                if col_name in summary_features_df.columns:
                    ordered_cols.append(col_name)

        for metric_key in summary_ce_like_metrics.keys():
            for feat in feature_order_ce_like:
                col_name = f'{metric_key}_{feat}'
                if col_name in summary_features_df.columns:
                    ordered_cols.append(col_name)

        remaining_cols = [c for c in summary_features_df.columns if c not in ordered_cols]
        summary_features_df = summary_features_df[ordered_cols + remaining_cols]
        summary_features_df.to_excel(writer, sheet_name='summary_features_ex1', index=False)

    if summary_c10_rows:
        summary_c10_df = pd.DataFrame(summary_c10_rows)

        fixed_cols = ['Barcode', 'Electrolyte']

        c10_snapshot_cols = [
            'Charge Capacity',
            'Discharge Capacity',
            'Charge Energy (mWh)',
            'Discharge Energy (mWh)',
            'Average Charge Voltage',
            'Average Discharge Voltage',
            'Coulombic Efficiency (%)',
            'RCE',
            'Charge Resistance',
            'Discharge Resistance',
            '100% ACR',
            'Energy Efficiency %',
            'Voltage Slippage',
            'V Polarization',
        ]

        soc50_cols = ['DCIR', 'DCIR0', 'DCIRt']

        ordered_cols = fixed_cols[:]

        # raw snapshots first
        for snap_name in ['f_1', 'f_2']:
            ordered_cols += [f'{snap_name} | C10 | {c}' for c in c10_snapshot_cols if f'{snap_name} | C10 | {c}' in summary_c10_df.columns]
            ordered_cols += [f'{snap_name} | SOC50 | {c}' for c in soc50_cols if f'{snap_name} | SOC50 | {c}' in summary_c10_df.columns]

        ordered_cols += [c for c in ['f_end | C10 | Map Corrected Cycle'] if c in summary_c10_df.columns]
        ordered_cols += [f'f_end | C10 | {c}' for c in c10_snapshot_cols if f'f_end | C10 | {c}' in summary_c10_df.columns]
        ordered_cols += [f'f_end | SOC50 | {c}' for c in soc50_cols if f'f_end | SOC50 | {c}' in summary_c10_df.columns]

        # delta blocks
        ordered_cols += [c for c in ['delta_1_to_2 | Kinetic Fading (%)'] if c in summary_c10_df.columns]
        for c10c in c10_snapshot_cols:
            for suffix in ['pct', 'pct_per_cyc']:
                col = f'delta_1_to_2 | C10 | {c10c} | {suffix}'
                if col in summary_c10_df.columns:
                    ordered_cols.append(col)

        ordered_cols += [c for c in ['delta_1_to_end | Kinetic Fading (%)'] if c in summary_c10_df.columns]
        for c10c in c10_snapshot_cols:
            for suffix in ['pct', 'pct_per_cyc']:
                col = f'delta_1_to_end | C10 | {c10c} | {suffix}'
                if col in summary_c10_df.columns:
                    ordered_cols.append(col)

        remaining_cols = [c for c in summary_c10_df.columns if c not in ordered_cols]
        summary_c10_df = summary_c10_df[ordered_cols + remaining_cols]

        summary_c10_df.to_excel(writer, sheet_name='summary_features_c10', index=False)

    if table_sheet_names:
        pd.DataFrame({'Tables': table_sheet_names}).to_excel(writer, sheet_name='Tables Index', index=False)
        ws = writer.book.add_worksheet('Tables Index Links')
        ws.write(0, 0, 'Open a table:')
        for r, name in enumerate(table_sheet_names, start=1):
            ws.write_url(r, 0, f"internal:'{name}'!A1", string=name)

print('---------------------------------------------------------------------------')
print('ALL DONE! Wow you are such a great scientist! :) Pat yourself on the back.')
print('Your file is saved to {}'.format(path))

In [ ]:
# Stage 1 ready pkl
# Saves a pickle next to the xlsx with the same per-cell summary features.
# Stage 1 extractor (stage1_2_pipeline) auto-loads this pkl when present
# and skips the heavy xlsx parse.
import pickle as _pkl, os as _os

_stage1_pkl_path = _os.path.splitext(path)[0] + '.stage1ready.pkl'
_stage1_payload = {}
if 'summary_features_df' in dir():
    _stage1_payload['summary_features_ex1'] = summary_features_df
if 'summary_c10_df' in dir():
    _stage1_payload['summary_features_c10'] = summary_c10_df

if _stage1_payload:
    with open(_stage1_pkl_path, 'wb') as _f:
        _pkl.dump(_stage1_payload, _f)
    print('Stage1-ready pkl saved:')
    print(' ', _stage1_pkl_path)
    print('  (Stage 1 of stage1_2_pipeline.ipynb will auto-detect this and load fast.)')
else:
    print('No summary_features dataframes in scope; pkl not written.')


In [ ]:
index_known

In [ ]:
metadata[metadata['Barcode']==cell]['Color'].values[0]

In [ ]:
metadata

In [ ]:
dcir_restbeforedcir_chg_600s

In [ ]:
dcir_soc50

In [ ]:
dcir_restbeforedcir_chg_600s


In [ ]:
charge_groups

In [ ]:
dcir_restbeforedcir_1200s

In [ ]:
temp_cycle_df

In [ ]:
print(f"cell: '{cell}'", type(cell))

In [ ]:
dcir_restbeforedcir_chg_600s

In [ ]:
corrected_cycles

In [ ]:
dV_discharged

In [ ]:
dis_rest_steps

In [ ]:
rest_discharged_steps 

In [ ]:
raw_data[(raw_data.Cycle==cycle) & (raw_data.Status.isin(['CC_Chg','CCCV_Chg']))]

In [ ]:
total_cap

In [ ]:
step_times

In [ ]:
statis_sub['Cycle']

In [ ]:
acr_chg_cycle

In [ ]:
cell

In [ ]:
dcir_soc50

In [ ]:
groups['DCIR0']

In [ ]:
groups

In [ ]:
groups_DCIR0_charge

In [ ]:
cc_chg_indices

In [ ]:
groups_DCIR0_rest_chg

In [ ]:
charge_groups

In [ ]:
 cc_chg_indices

In [ ]:
dcir_soc50

In [ ]:
dcir_soc50_chg

In [ ]:
chart_soc50_dcirt